# OA Tribe Model: K Comparison, Assignment Maps, and Scorecard

This notebook does three jobs:

1. Re-runs clustering for K=5 to K=10 from `oa_cluster_features_v1.csv`.
2. Exports one OA-level assignment file per K, e.g. `k6_oa_cluster_assignments_v1.csv`.
3. Compares K runs using size balance, z-score distinctiveness, near-duplicate detection, K-to-K split lineage, geography summaries, optional seed stability, and a final scorecard.

Important: if you did not save OA-level labels from your previous clustering runs, they cannot be reconstructed from summary CSVs alone. This notebook re-runs the clustering and saves the assignment files properly.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score
from sklearn.metrics.pairwise import cosine_similarity

try:
    import joblib
except ImportError:
    joblib = None

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

## 1. Configure paths

Change these paths to match your project.

Expected input:
- `oa_cluster_features_v1.csv`
- Optional geography lookup containing OA code, LAD code and ward code
- Existing K summary CSVs, if already generated

Expected outputs:
- `k5_oa_cluster_assignments_v1.csv` through `k10_oa_cluster_assignments_v1.csv`
- K comparison tables
- K-to-K lineage tables
- Final model scorecard

In [2]:
# You are running this notebook from:
# C:\Users\keena\Documents\Electoral_Tribes\notebooks
#
# Project root should therefore be:
# C:\Users\keena\Documents\Electoral_Tribes

from pathlib import Path

NOTEBOOK_DIR = Path.cwd()

# If the notebook is inside a folder called "notebooks", move one level up.
# Otherwise, assume the current folder is already the project root.
if NOTEBOOK_DIR.name.lower() == "notebooks":
    PROJECT_DIR = NOTEBOOK_DIR.parent
else:
    PROJECT_DIR = NOTEBOOK_DIR

DATA_PROCESSED_DIR = PROJECT_DIR / "data" / "processed"

# Main model input
FEATURES_PATH = DATA_PROCESSED_DIR / "oa_cluster_features_v1.csv"

# No geography lookup for now.
# Assignment files will include OA code, cluster ID and population only.
# LAD / ward fields can be added later when you have an OA geography lookup.
GEOGRAPHY_LOOKUP_PATH = None

# Existing K summary files.
# This should point to the folder containing files such as:
# k5_cluster_feature_zscores_v1.csv
# k5_cluster_sizes_v1.csv
SUMMARY_DIR = DATA_PROCESSED_DIR

# Output folder
OUTPUT_DIR = DATA_PROCESSED_DIR / "k_comparison_outputs_v1"
ASSIGNMENT_DIR = OUTPUT_DIR / "oa_assignments"
MODEL_DIR = OUTPUT_DIR / "models"
REPORT_DIR = OUTPUT_DIR / "reports"

for d in [OUTPUT_DIR, ASSIGNMENT_DIR, MODEL_DIR, REPORT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

K_VALUES = list(range(5, 11))
RANDOM_STATE = 42
N_INIT = 50

print("Notebook folder:", NOTEBOOK_DIR)
print("Project folder:", PROJECT_DIR)
print("Processed data folder:", DATA_PROCESSED_DIR)
print("Feature file:", FEATURES_PATH)
print("Summary folder:", SUMMARY_DIR)
print("Output folder:", OUTPUT_DIR)

if not FEATURES_PATH.exists():
    raise FileNotFoundError(f"Feature file not found: {FEATURES_PATH}")


Notebook folder: c:\Users\keena\Documents\Electoral_Tribes\notebooks
Project folder: c:\Users\keena\Documents\Electoral_Tribes
Processed data folder: c:\Users\keena\Documents\Electoral_Tribes\data\processed
Feature file: c:\Users\keena\Documents\Electoral_Tribes\data\processed\oa_cluster_features_v1.csv
Summary folder: c:\Users\keena\Documents\Electoral_Tribes\data\processed
Output folder: c:\Users\keena\Documents\Electoral_Tribes\data\processed\k_comparison_outputs_v1


## 2. Helper functions

These functions detect common column names and keep the notebook tolerant of small naming differences.

In [3]:
def first_existing(columns, candidates, required=True):
    cols = list(columns)
    lower_map = {c.lower(): c for c in cols}

    for cand in candidates:
        if cand in cols:
            return cand
        if cand.lower() in lower_map:
            return lower_map[cand.lower()]

    if required:
        raise KeyError(f"None of these candidate columns were found: {candidates}")
    return None


def normalise_code_columns(df):
    df = df.copy()
    rename = {}

    oa_col = first_existing(df.columns, ["OA21CD", "oa_code", "OA_CODE", "geography code", "geography_code"], required=False)
    if oa_col is not None:
        rename[oa_col] = "oa_code"

    pop_col = first_existing(df.columns, ["population", "total_residents", "Total_Residents", "MSOA_POPULATION"], required=False)
    if pop_col is not None:
        rename[pop_col] = "population"

    lad_col = first_existing(df.columns, ["LAD25CD", "LAD26CD", "lad_code", "LAD_CODE"], required=False)
    if lad_col is not None:
        rename[lad_col] = "lad_code"

    lad_name_col = first_existing(df.columns, ["LAD25NM", "LAD26NM", "lad_name", "LAD_NAME"], required=False)
    if lad_name_col is not None:
        rename[lad_name_col] = "lad_name"

    ward_col = first_existing(df.columns, ["WD25CD", "WD26CD", "ward_code", "WARD_CODE"], required=False)
    if ward_col is not None:
        rename[ward_col] = "ward_code"

    ward_name_col = first_existing(df.columns, ["WD25NM", "WD26NM", "ward_name", "WARD_NAME"], required=False)
    if ward_name_col is not None:
        rename[ward_name_col] = "ward_name"

    df = df.rename(columns=rename)
    return df


def infer_feature_columns(df):
    non_features = {
        "oa_code", "oa_name", "OA21CD", "OA21NM",
        "population", "total_residents",
        "lad_code", "lad_name", "ward_code", "ward_name",
        "cluster_id"
    }

    feature_cols = []
    for col in df.columns:
        if col in non_features:
            continue
        if pd.api.types.is_numeric_dtype(df[col]):
            feature_cols.append(col)

    return feature_cols


def safe_share(n, d):
    return np.where(d > 0, n / d, np.nan)


def label_quality_from_share(x, good_threshold, ok_threshold, lower_is_better=False):
    if pd.isna(x):
        return "not assessed"

    if lower_is_better:
        if x <= good_threshold:
            return "good"
        elif x <= ok_threshold:
            return "acceptable"
        else:
            return "weak"
    else:
        if x >= good_threshold:
            return "good"
        elif x >= ok_threshold:
            return "acceptable"
        else:
            return "weak"

## 3. Load OA feature matrix

This uses your `oa_cluster_features_v1.csv`.

The notebook assumes:
- one row per OA
- one OA code column
- one population column
- numeric feature columns

In [4]:
oa_features = pd.read_csv(FEATURES_PATH, low_memory=False)
oa_features = normalise_code_columns(oa_features)

if "oa_code" not in oa_features.columns:
    raise KeyError("Could not detect OA code column. Rename your OA column to OA21CD or oa_code.")

if "population" not in oa_features.columns:
    print("No population column detected. Setting population = 1 for unweighted comparisons.")
    oa_features["population"] = 1

oa_features["oa_code"] = oa_features["oa_code"].astype(str).str.strip()
oa_features["population"] = pd.to_numeric(oa_features["population"], errors="coerce").fillna(0)

feature_cols = infer_feature_columns(oa_features)

print("Rows:", len(oa_features))
print("Unique OAs:", oa_features["oa_code"].nunique())
print("Feature columns:", len(feature_cols))
print(feature_cols)
display(oa_features.head())

Rows: 188880
Unique OAs: 188880
Feature columns: 32
['age_15_24_pct', 'age_25_34_pct', 'age_50_64_pct', 'age_65_plus_pct', 'uk_born_pct', 'non_uk_born_pct', 'resident_10_plus_years_pct', 'resident_less_5_years_pct', 'white_british_pct', 'white_other_pct', 'non_white_pct', 'owned_pct', 'owns_outright_pct', 'social_rented_pct', 'private_rented_pct', 'house_type_pct', 'flat_type_pct', 'managerial_professional_pct', 'skilled_traditional_pct', 'routine_service_elementary_pct', 'employed_pct', 'unemployed_pct', 'full_time_student_pct', 'retired_pct', 'long_term_sick_disabled_pct', 'no_qualifications_pct', 'level_1_2_pct', 'apprenticeship_pct', 'level_4_plus_pct', 'one_person_household_pct', 'married_couple_family_pct', 'lone_parent_family_pct']


,oa_code,population,age_15_24_pct,age_25_34_pct,age_50_64_pct,age_65_plus_pct,uk_born_pct,non_uk_born_pct,resident_10_plus_years_pct,resident_less_5_years_pct,white_british_pct,white_other_pct,non_white_pct,owned_pct,owns_outright_pct,social_rented_pct,private_rented_pct,house_type_pct,flat_type_pct,managerial_professional_pct,skilled_traditional_pct,routine_service_elementary_pct,employed_pct,unemployed_pct,full_time_student_pct,retired_pct,long_term_sick_disabled_pct,no_qualifications_pct,level_1_2_pct,apprenticeship_pct,level_4_plus_pct,one_person_household_pct,married_couple_family_pct,lone_parent_family_pct
0,E00000001,176,0.039773,0.096591,0.227273,0.380682,0.670455,0.329545,0.193182,0.113636,0.636364,0.221591,0.119318,0.765957,0.627660,0.074468,0.159574,0.010638,0.989362,0.840000,0.022989,0.064103,0.546584,0.062112,0.031056,0.322981,0.006211,0.029304,0.106250,0.006250,0.754386,0.361702,0.329787,0.053191
1,E00000003,256,0.089844,0.117188,0.203125,0.296875,0.715953,0.284047,0.167969,0.050781,0.682353,0.137255,0.180392,0.819820,0.540541,0.018018,0.153153,0.110092,0.889908,0.840000,0.021898,0.064103,0.605381,0.022422,0.089686,0.255605,0.000000,0.029304,0.057413,0.004525,0.754386,0.247706,0.394495,0.027523
2,E00000005,112,0.080357,0.125000,0.250000,0.250000,0.657658,0.342342,0.241071,0.062500,0.580357,0.178571,0.241071,0.634921,0.460317,0.031746,0.333333,0.015873,0.984127,0.840000,0.019868,0.064103,0.628571,0.028571,0.038095,0.219048,0.009524,0.029304,0.057413,0.028302,0.754386,0.412698,0.238095,0.063492
3,E00000007,144,0.118056,0.375000,0.118056,0.118056,0.408163,0.591837,0.131034,0.283111,0.430556,0.277778,0.263889,0.321839,0.229885,0.011494,0.655172,0.023256,0.976744,0.840000,0.047170,0.064103,0.767606,0.007042,0.077465,0.119718,0.007042,0.035971,0.057413,0.007194,0.754386,0.430233,0.116279,0.008065
4,E00000010,178,0.073034,0.185393,0.224719,0.117978,0.541899,0.458101,0.269663,0.089888,0.474286,0.211429,0.314286,0.198413,0.103175,0.555556,0.246032,0.006667,0.992806,0.692982,0.061404,0.175439,0.652941,0.105882,0.052941,0.088235,0.052941,0.087209,0.191860,0.000000,0.540698,0.666667,0.069737,0.008065


## 4. Attach LAD and ward codes, if available

If your `oa_cluster_features_v1.csv` already contains `lad_code` and `ward_code`, this step keeps them.

If not, it tries to load `oa21_to_ward25_lad25.csv`.

In [ ]:
has_geo = {"lad_code", "ward_code"}.issubset(set(oa_features.columns))

if not has_geo and GEOGRAPHY_LOOKUP_PATH is not None and Path(GEOGRAPHY_LOOKUP_PATH).exists():
    geo = pd.read_csv(GEOGRAPHY_LOOKUP_PATH, low_memory=False)
    geo = normalise_code_columns(geo)

    keep_cols = [c for c in ["oa_code", "lad_code", "lad_name", "ward_code", "ward_name"] if c in geo.columns]
    geo = geo[keep_cols].drop_duplicates("oa_code")

    oa_features = oa_features.merge(
        geo,
        on="oa_code",
        how="left",
        validate="one_to_one"
    )

    print("Geography lookup joined.")
elif has_geo:
    print("Feature file already contains LAD / ward columns.")
else:
    print("No geography lookup being used. Assignment files will include OA code, cluster ID and population only.")

geo_cols = [c for c in ["lad_code", "lad_name", "ward_code", "ward_name"] if c in oa_features.columns]
print("Geography columns available:", geo_cols)


## 5. Preprocess features

This must match the preprocessing used in your earlier cluster summaries.

If your previous clustering used a different scaler, clipping rule or random seed, the new assignments may not exactly match the previous summary outputs.

For a clean reproducible system, re-run everything from this notebook and treat these as the canonical V1 outputs.

In [5]:
X = oa_features[feature_cols].copy()

# Numeric coercion
for col in feature_cols:
    X[col] = pd.to_numeric(X[col], errors="coerce")

# Median fill
feature_medians = X.median(numeric_only=True)
X = X.fillna(feature_medians)

# Clip extreme values to reduce tiny-OA noise
for col in feature_cols:
    lower = X[col].quantile(0.01)
    upper = X[col].quantile(0.99)
    X[col] = X[col].clip(lower, upper)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Scaled matrix:", X_scaled.shape)

Scaled matrix: (188880, 32)


## 6. Run K=5 to K=10 and export OA assignment files

Each output file contains:

- `oa_code`
- `cluster_id`
- `population`
- optional `lad_code`, `lad_name`, `ward_code`, `ward_name`

In [6]:
assignment_frames = {}

base_cols = ["oa_code", "population"] + [c for c in ["lad_code", "lad_name", "ward_code", "ward_name"] if c in oa_features.columns]

for k in K_VALUES:
    print(f"Running K={k}")

    model = KMeans(
        n_clusters=k,
        random_state=RANDOM_STATE,
        n_init=N_INIT,
        algorithm="lloyd"
    )

    labels = model.fit_predict(X_scaled)

    assignments = oa_features[base_cols].copy()
    assignments["cluster_id"] = labels

    # Put cluster_id after oa_code
    ordered_cols = ["oa_code", "cluster_id", "population"] + [c for c in base_cols if c not in ["oa_code", "population"]]
    assignments = assignments[ordered_cols]

    out_path = ASSIGNMENT_DIR / f"k{k}_oa_cluster_assignments_v1.csv"
    assignments.to_csv(out_path, index=False)

    assignment_frames[k] = assignments

    if joblib is not None:
        joblib.dump(model, MODEL_DIR / f"k{k}_kmeans_model_v1.joblib")

    print("Saved:", out_path)

if joblib is not None:
    joblib.dump(scaler, MODEL_DIR / "standard_scaler_v1.joblib")
    print("Saved scaler.")

Running K=5
Saved: c:\Users\keena\Documents\Electoral_Tribes\data\processed\k_comparison_outputs_v1\oa_assignments\k5_oa_cluster_assignments_v1.csv
Running K=6
Saved: c:\Users\keena\Documents\Electoral_Tribes\data\processed\k_comparison_outputs_v1\oa_assignments\k6_oa_cluster_assignments_v1.csv
Running K=7
Saved: c:\Users\keena\Documents\Electoral_Tribes\data\processed\k_comparison_outputs_v1\oa_assignments\k7_oa_cluster_assignments_v1.csv
Running K=8
Saved: c:\Users\keena\Documents\Electoral_Tribes\data\processed\k_comparison_outputs_v1\oa_assignments\k8_oa_cluster_assignments_v1.csv
Running K=9
Saved: c:\Users\keena\Documents\Electoral_Tribes\data\processed\k_comparison_outputs_v1\oa_assignments\k9_oa_cluster_assignments_v1.csv
Running K=10
Saved: c:\Users\keena\Documents\Electoral_Tribes\data\processed\k_comparison_outputs_v1\oa_assignments\k10_oa_cluster_assignments_v1.csv
Saved scaler.


## 7. Generate cluster sizes from assignment files

This is a sanity check and a useful comparison table.

In [7]:
size_rows = []

for k, df in assignment_frames.items():
    s = (
        df.groupby("cluster_id", as_index=False)
        .agg(
            oa_count=("oa_code", "count"),
            population=("population", "sum")
        )
    )

    s["K"] = k
    s["oa_share"] = s["oa_count"] / s["oa_count"].sum()
    s["population_share"] = s["population"] / s["population"].sum()

    size_rows.append(s)

cluster_sizes_from_assignments = pd.concat(size_rows, ignore_index=True)
cluster_sizes_from_assignments = cluster_sizes_from_assignments[
    ["K", "cluster_id", "oa_count", "population", "oa_share", "population_share"]
]

display(cluster_sizes_from_assignments)
cluster_sizes_from_assignments.to_csv(REPORT_DIR / "cluster_sizes_from_assignments_v1.csv", index=False)

,K,cluster_id,oa_count,population,oa_share,population_share
0,5,0,22954,8680492,0.121527,0.145651
1,5,1,46508,13891185,0.246230,0.233082
2,5,2,44169,14851322,0.233847,0.249193
3,5,3,16747,4983020,0.088665,0.083611
4,5,4,58502,17191728,0.309731,0.288463
5,6,0,45178,15141521,0.239189,0.254062
6,6,1,56674,16597100,0.300053,0.278485
7,6,2,21385,7910039,0.113220,0.132724
8,6,3,46299,13833605,0.245124,0.232116
9,6,4,15818,4601819,0.083746,0.077215


## 8. Load existing K summary files

This expects files named like:

- `k5_cluster_feature_zscores_v1.csv`
- `k5_cluster_defining_features_v1.csv`
- `k5_cluster_profile_summaries_v1.csv`
- `k5_cluster_sizes_v1.csv`

If these files are not in `SUMMARY_DIR`, either move them there or change `SUMMARY_DIR`.

In [8]:
def load_k_file(k, suffix, required=False):
    path = SUMMARY_DIR / f"k{k}_{suffix}_v1.csv"
    if not path.exists():
        if required:
            raise FileNotFoundError(path)
        print(f"Missing: {path.name}")
        return None
    return pd.read_csv(path)

summary = {}

for k in K_VALUES:
    summary[k] = {
        "zscores": load_k_file(k, "cluster_feature_zscores"),
        "defining": load_k_file(k, "cluster_defining_features"),
        "profiles": load_k_file(k, "cluster_profile_summaries"),
        "sizes": load_k_file(k, "cluster_sizes"),
        "means": load_k_file(k, "cluster_feature_means"),
        "medians": load_k_file(k, "cluster_feature_medians"),
        "age": load_k_file(k, "cluster_age_band_shares"),
    }

print("Summary files loaded.")

Summary files loaded.


## 9. Z-score distinctiveness comparison

Good clusters usually have a clear profile: multiple features with high positive/negative z-scores.

This produces:
- average absolute z-score per cluster
- max absolute z-score per cluster
- number of strongly defining features per cluster

In [9]:
distinctiveness_rows = []

for k in K_VALUES:
    z = summary[k]["zscores"]
    if z is None:
        continue

    z = z.copy()
    z["K"] = k

    z_feature_cols = [c for c in z.columns if c not in ["K", "cluster_id"]]

    for _, row in z.iterrows():
        vals = row[z_feature_cols].astype(float)
        distinctiveness_rows.append({
            "K": k,
            "cluster_id": row["cluster_id"],
            "mean_abs_z": vals.abs().mean(),
            "max_abs_z": vals.abs().max(),
            "n_features_abs_z_ge_0_75": int((vals.abs() >= 0.75).sum()),
            "n_features_abs_z_ge_1_00": int((vals.abs() >= 1.00).sum()),
            "n_features_abs_z_ge_1_50": int((vals.abs() >= 1.50).sum()),
        })

cluster_distinctiveness = pd.DataFrame(distinctiveness_rows)
display(cluster_distinctiveness)

k_distinctiveness_summary = (
    cluster_distinctiveness
    .groupby("K", as_index=False)
    .agg(
        avg_mean_abs_z=("mean_abs_z", "mean"),
        avg_max_abs_z=("max_abs_z", "mean"),
        min_defining_features_075=("n_features_abs_z_ge_0_75", "min"),
        avg_defining_features_075=("n_features_abs_z_ge_0_75", "mean"),
        min_defining_features_100=("n_features_abs_z_ge_1_00", "min"),
        avg_defining_features_100=("n_features_abs_z_ge_1_00", "mean"),
    )
)

display(k_distinctiveness_summary)

cluster_distinctiveness.to_csv(REPORT_DIR / "cluster_distinctiveness_v1.csv", index=False)
k_distinctiveness_summary.to_csv(REPORT_DIR / "k_distinctiveness_summary_v1.csv", index=False)

,K,cluster_id,mean_abs_z,max_abs_z,n_features_abs_z_ge_0_75,n_features_abs_z_ge_1_00,n_features_abs_z_ge_1_50
0,5,0.0,0.469867,1.090061,7,1,0
1,5,1.0,0.375230,0.855897,3,0,0
2,5,2.0,1.120019,1.875431,24,18,10
3,5,3.0,0.518555,1.033324,4,2,0
4,5,4.0,0.823898,1.934990,17,8,5
5,6,0.0,0.388810,0.875137,3,0,0
6,6,1.0,0.518434,1.037292,4,3,0
7,6,2.0,0.801150,2.037254,15,7,5
8,6,3.0,0.470222,1.090336,7,1,0
9,6,4.0,1.073810,1.916411,24,17,10


,K,avg_mean_abs_z,avg_max_abs_z,min_defining_features_075,avg_defining_features_075,min_defining_features_100,avg_defining_features_100
0,5,0.661514,1.357941,3,11.000000,0,5.800000
1,6,0.746177,2.062281,3,12.166667,0,7.166667
2,7,0.716592,1.984878,0,11.857143,0,7.428571
3,8,0.719180,1.947746,0,11.750000,0,6.875000
4,9,0.714873,1.953951,0,11.444444,0,7.000000
5,10,0.737785,1.959723,0,11.900000,0,7.400000


## 10. Duplicate / near-duplicate cluster detection

This compares cluster z-score profiles within each K.

High cosine similarity means two clusters have very similar profiles.

Treat this as a warning flag, not an automatic rejection. Sometimes two clusters differ geographically or politically despite similar demographic profiles.

In [10]:
similarity_rows = []
near_duplicate_rows = []

NEAR_DUPLICATE_THRESHOLD = 0.92

for k in K_VALUES:
    z = summary[k]["zscores"]
    if z is None:
        continue

    z = z.sort_values("cluster_id").copy()
    z_feature_cols = [c for c in z.columns if c != "cluster_id"]

    matrix = z[z_feature_cols].astype(float).to_numpy()
    sims = cosine_similarity(matrix)

    cluster_ids = z["cluster_id"].tolist()

    for i in range(len(cluster_ids)):
        for j in range(i + 1, len(cluster_ids)):
            row = {
                "K": k,
                "cluster_a": cluster_ids[i],
                "cluster_b": cluster_ids[j],
                "cosine_similarity": sims[i, j],
            }
            similarity_rows.append(row)

            if sims[i, j] >= NEAR_DUPLICATE_THRESHOLD:
                near_duplicate_rows.append(row)

cluster_similarity = pd.DataFrame(similarity_rows)
near_duplicates = pd.DataFrame(near_duplicate_rows)

closest_similarity_by_k = (
    cluster_similarity
    .groupby("K", as_index=False)
    .agg(closest_cluster_similarity=("cosine_similarity", "max"))
)

display(closest_similarity_by_k)
display(near_duplicates)

cluster_similarity.to_csv(REPORT_DIR / "cluster_pairwise_similarity_v1.csv", index=False)
near_duplicates.to_csv(REPORT_DIR / "near_duplicate_clusters_v1.csv", index=False)
closest_similarity_by_k.to_csv(REPORT_DIR / "closest_similarity_by_k_v1.csv", index=False)

,K,closest_cluster_similarity
0,5,0.589027
1,6,0.550835
2,7,0.549889
3,8,0.670982
4,9,0.695718
5,10,0.739004


""


## 11. K-to-K split lineage

This is the main test for whether K+1 adds a meaningful new tribe or merely splits an existing one.

For each adjacent pair:
- K=5 to K=6
- K=6 to K=7
- ...
- K=9 to K=10

The lineage table shows how each parent cluster splits into child clusters.

In [11]:
lineage_tables = []
lineage_summary_rows = []

for parent_k, child_k in zip(K_VALUES[:-1], K_VALUES[1:]):
    parent = assignment_frames[parent_k][["oa_code", "cluster_id", "population"]].rename(
        columns={"cluster_id": "parent_cluster", "population": "population_parent"}
    )
    child = assignment_frames[child_k][["oa_code", "cluster_id", "population"]].rename(
        columns={"cluster_id": "child_cluster", "population": "population_child"}
    )

    merged = parent.merge(child, on="oa_code", how="inner", validate="one_to_one")
    merged["population"] = merged["population_child"].fillna(merged["population_parent"])

    cross = (
        merged.groupby(["parent_cluster", "child_cluster"], as_index=False)
        .agg(population=("population", "sum"), oa_count=("oa_code", "count"))
    )

    parent_totals = cross.groupby("parent_cluster", as_index=False)["population"].sum().rename(
        columns={"population": "parent_population"}
    )
    child_totals = cross.groupby("child_cluster", as_index=False)["population"].sum().rename(
        columns={"population": "child_population"}
    )

    cross = cross.merge(parent_totals, on="parent_cluster", how="left")
    cross = cross.merge(child_totals, on="child_cluster", how="left")

    cross["share_of_parent"] = cross["population"] / cross["parent_population"]
    cross["share_of_child"] = cross["population"] / cross["child_population"]
    cross["parent_k"] = parent_k
    cross["child_k"] = child_k

    cross = cross[
        ["parent_k", "child_k", "parent_cluster", "child_cluster", "oa_count", "population",
         "share_of_parent", "share_of_child", "parent_population", "child_population"]
    ]

    lineage_tables.append(cross)

    # Parent split diagnostics
    parent_diag = (
        cross.sort_values(["parent_cluster", "share_of_parent"], ascending=[True, False])
        .groupby("parent_cluster")
        .agg(
            largest_child_share=("share_of_parent", "max"),
            n_children_over_10pct=("share_of_parent", lambda x: int((x >= 0.10).sum())),
            n_children_over_20pct=("share_of_parent", lambda x: int((x >= 0.20).sum())),
        )
        .reset_index()
    )

    # A meaningful split is roughly: no single child keeps nearly all of parent,
    # and at least two children take material shares.
    meaningful_split_count = int(((parent_diag["largest_child_share"] <= 0.80) & (parent_diag["n_children_over_20pct"] >= 2)).sum())

    ari = adjusted_rand_score(merged["parent_cluster"], merged["child_cluster"])
    nmi = normalized_mutual_info_score(merged["parent_cluster"], merged["child_cluster"])

    lineage_summary_rows.append({
        "parent_k": parent_k,
        "child_k": child_k,
        "adjusted_rand_index": ari,
        "normalised_mutual_info": nmi,
        "meaningful_parent_splits": meaningful_split_count,
        "parents_total": parent_diag["parent_cluster"].nunique(),
        "max_largest_child_share": parent_diag["largest_child_share"].max(),
        "median_largest_child_share": parent_diag["largest_child_share"].median(),
    })

    out_path = REPORT_DIR / f"k{parent_k}_to_k{child_k}_lineage_v1.csv"
    cross.to_csv(out_path, index=False)
    print("Saved:", out_path)

lineage_all = pd.concat(lineage_tables, ignore_index=True)
lineage_summary = pd.DataFrame(lineage_summary_rows)

display(lineage_summary)

lineage_all.to_csv(REPORT_DIR / "k_to_k_lineage_all_v1.csv", index=False)
lineage_summary.to_csv(REPORT_DIR / "k_to_k_lineage_summary_v1.csv", index=False)

Saved: c:\Users\keena\Documents\Electoral_Tribes\data\processed\k_comparison_outputs_v1\reports\k5_to_k6_lineage_v1.csv
Saved: c:\Users\keena\Documents\Electoral_Tribes\data\processed\k_comparison_outputs_v1\reports\k6_to_k7_lineage_v1.csv
Saved: c:\Users\keena\Documents\Electoral_Tribes\data\processed\k_comparison_outputs_v1\reports\k7_to_k8_lineage_v1.csv
Saved: c:\Users\keena\Documents\Electoral_Tribes\data\processed\k_comparison_outputs_v1\reports\k8_to_k9_lineage_v1.csv
Saved: c:\Users\keena\Documents\Electoral_Tribes\data\processed\k_comparison_outputs_v1\reports\k9_to_k10_lineage_v1.csv


,parent_k,child_k,adjusted_rand_index,normalised_mutual_info,meaningful_parent_splits,parents_total,max_largest_child_share,median_largest_child_share
0,5,6,0.938880,0.910405,0,5,0.989323,0.961823
1,6,7,0.576363,0.704023,2,6,0.994694,0.843862
2,7,8,0.638300,0.732880,3,7,0.974570,0.799475
3,8,9,0.757392,0.806455,1,8,0.995159,0.913859
4,9,10,0.919085,0.901844,1,9,0.990591,0.973412


## 12. Geography summary

This does not replace maps, but it helps you see whether clusters are geographically coherent.

If `ward_code` is present, it calculates how dominant the largest cluster is within each ward.

In [ ]:
geography_summary_rows = []

for k, df in assignment_frames.items():
    if "ward_code" not in df.columns:
        continue

    ward_cluster = (
        df.groupby(["ward_code", "cluster_id"], as_index=False)
        .agg(population=("population", "sum"), oa_count=("oa_code", "count"))
    )

    ward_totals = (
        ward_cluster.groupby("ward_code", as_index=False)
        .agg(ward_population=("population", "sum"))
    )

    ward_cluster = ward_cluster.merge(ward_totals, on="ward_code", how="left")
    ward_cluster["cluster_share_in_ward"] = ward_cluster["population"] / ward_cluster["ward_population"]

    ward_dominance = (
        ward_cluster.groupby("ward_code", as_index=False)
        .agg(dominant_cluster_share=("cluster_share_in_ward", "max"))
    )

    geography_summary_rows.append({
        "K": k,
        "wards_available": ward_dominance["ward_code"].nunique(),
        "mean_ward_dominant_cluster_share": ward_dominance["dominant_cluster_share"].mean(),
        "median_ward_dominant_cluster_share": ward_dominance["dominant_cluster_share"].median(),
        "wards_with_60pct_plus_dominant_cluster": int((ward_dominance["dominant_cluster_share"] >= 0.60).sum()),
        "wards_with_75pct_plus_dominant_cluster": int((ward_dominance["dominant_cluster_share"] >= 0.75).sum()),
    })

    ward_dominance.to_csv(REPORT_DIR / f"k{k}_ward_cluster_dominance_v1.csv", index=False)

geography_summary = pd.DataFrame(geography_summary_rows)
display(geography_summary)

if len(geography_summary):
    geography_summary.to_csv(REPORT_DIR / "k_geography_summary_v1.csv", index=False)

## 13. Optional: random seed stability

This re-runs each K across multiple seeds and compares assignments.

High ARI means similar cluster structures across random starts.

This cell may take a few minutes.

In [13]:
RUN_STABILITY = True

if RUN_STABILITY:
    SEEDS = [1, 7, 13, 21, 42, 99, 123]
    stability_rows = []

    for k in K_VALUES:
        seed_labels = {}

        for seed in SEEDS:
            model = KMeans(
                n_clusters=k,
                random_state=seed,
                n_init=N_INIT,
                algorithm="lloyd"
            )
            seed_labels[seed] = model.fit_predict(X_scaled)

        for i, seed_a in enumerate(SEEDS):
            for seed_b in SEEDS[i+1:]:
                stability_rows.append({
                    "K": k,
                    "seed_a": seed_a,
                    "seed_b": seed_b,
                    "adjusted_rand_index": adjusted_rand_score(seed_labels[seed_a], seed_labels[seed_b]),
                    "normalised_mutual_info": normalized_mutual_info_score(seed_labels[seed_a], seed_labels[seed_b]),
                })

    stability = pd.DataFrame(stability_rows)

    k_stability_summary = (
        stability.groupby("K", as_index=False)
        .agg(
            mean_ari=("adjusted_rand_index", "mean"),
            min_ari=("adjusted_rand_index", "min"),
            mean_nmi=("normalised_mutual_info", "mean"),
            min_nmi=("normalised_mutual_info", "min"),
        )
    )

    display(k_stability_summary)

    stability.to_csv(REPORT_DIR / "seed_stability_pairwise_v1.csv", index=False)
    k_stability_summary.to_csv(REPORT_DIR / "k_seed_stability_summary_v1.csv", index=False)
else:
    print("Seed stability skipped. Set RUN_STABILITY = True to run it.")

,K,mean_ari,min_ari,mean_nmi,min_nmi
0,5,0.994437,0.990677,0.990150,0.984356
1,6,0.990540,0.984571,0.986744,0.979263
2,7,0.996727,0.993972,0.994838,0.991000
3,8,0.993767,0.988375,0.990873,0.984908
4,9,0.986822,0.976435,0.983587,0.973495
5,10,0.988732,0.982256,0.985046,0.978148


## 14. Final scorecard

This creates an automatic comparison table.

Political usefulness is marked as `not assessed` unless you later add vote/member validation data.
Interpretability is partly automatic but should be manually reviewed after reading the cluster summaries and maps.

In [14]:
scorecard = cluster_sizes_from_assignments.groupby("K", as_index=False).agg(
    min_cluster_population_share=("population_share", "min"),
    max_cluster_population_share=("population_share", "max"),
    min_cluster_oa_share=("oa_share", "min"),
    max_cluster_oa_share=("oa_share", "max"),
)

scorecard["cluster_size_ratio_pop"] = (
    scorecard["max_cluster_population_share"] / scorecard["min_cluster_population_share"]
)

if "k_distinctiveness_summary" in globals() and len(k_distinctiveness_summary):
    scorecard = scorecard.merge(k_distinctiveness_summary, on="K", how="left")

if "closest_similarity_by_k" in globals() and len(closest_similarity_by_k):
    scorecard = scorecard.merge(closest_similarity_by_k, on="K", how="left")

if "geography_summary" in globals() and len(geography_summary):
    scorecard = scorecard.merge(geography_summary, on="K", how="left")

# Add lineage / split value for K>5 from previous K
split_value = lineage_summary.rename(columns={"child_k": "K"})[
    ["K", "adjusted_rand_index", "normalised_mutual_info", "meaningful_parent_splits", "median_largest_child_share"]
]
scorecard = scorecard.merge(split_value, on="K", how="left")

# Qualitative labels
scorecard["size_balance"] = scorecard["min_cluster_population_share"].apply(
    lambda x: label_quality_from_share(x, good_threshold=0.05, ok_threshold=0.025, lower_is_better=False)
)

scorecard["zscore_separation"] = scorecard["avg_defining_features_075"].apply(
    lambda x: label_quality_from_share(x, good_threshold=6, ok_threshold=4, lower_is_better=False)
)

scorecard["closest_similarity_rating"] = scorecard["closest_cluster_similarity"].apply(
    lambda x: label_quality_from_share(x, good_threshold=0.80, ok_threshold=0.90, lower_is_better=True)
)

scorecard["split_value"] = scorecard.apply(
    lambda r: "baseline" if r["K"] == min(K_VALUES)
    else ("good" if pd.notna(r.get("meaningful_parent_splits")) and r["meaningful_parent_splits"] >= 1
          else "weak_or_unclear"),
    axis=1
)

scorecard["political_usefulness"] = "not assessed"
scorecard["interpretability_manual_review"] = "review cluster summaries and maps"

# Simple numeric score
def score_label(label):
    return {"good": 2, "acceptable": 1, "weak": 0, "baseline": 1, "weak_or_unclear": 0, "not assessed": np.nan}.get(label, np.nan)

scorecard["auto_score"] = (
    scorecard["size_balance"].map(score_label).fillna(0)
    + scorecard["zscore_separation"].map(score_label).fillna(0)
    + scorecard["closest_similarity_rating"].map(score_label).fillna(0)
    + scorecard["split_value"].map(score_label).fillna(0)
)

scorecard["verdict"] = np.select(
    [
        scorecard["K"].eq(5),
        (scorecard["auto_score"] >= 6) & (scorecard["size_balance"] != "weak"),
        (scorecard["auto_score"] >= 4) & (scorecard["closest_similarity_rating"] != "weak"),
        scorecard["auto_score"] < 4,
    ],
    [
        "baseline",
        "strong candidate",
        "challenger / needs review",
        "probably reject unless maps show exceptional value",
    ],
    default="manual review"
)

display(scorecard)

scorecard.to_csv(REPORT_DIR / "k_model_scorecard_v1.csv", index=False)

,K,min_cluster_population_share,max_cluster_population_share,min_cluster_oa_share,max_cluster_oa_share,cluster_size_ratio_pop,avg_mean_abs_z,avg_max_abs_z,min_defining_features_075,avg_defining_features_075,min_defining_features_100,avg_defining_features_100,closest_cluster_similarity,adjusted_rand_index,normalised_mutual_info,meaningful_parent_splits,median_largest_child_share,size_balance,zscore_separation,closest_similarity_rating,split_value,political_usefulness,interpretability_manual_review,auto_score,verdict
0,5,0.083611,0.288463,0.088665,0.309731,3.450062,0.661514,1.357941,3,11.000000,0,5.800000,0.589027,NaN,NaN,NaN,NaN,good,good,good,baseline,not assessed,review cluster summaries and maps,7,baseline
1,6,0.025398,0.278485,0.018668,0.300053,10.964858,0.746177,2.062281,3,12.166667,0,7.166667,0.550835,0.938880,0.910405,0.0,0.961823,acceptable,good,good,weak_or_unclear,not assessed,review cluster summaries and maps,5,challenger / needs review
2,7,0.025427,0.220961,0.018763,0.226917,8.690012,0.716592,1.984878,0,11.857143,0,7.428571,0.549889,0.576363,0.704023,2.0,0.843862,acceptable,good,good,good,not assessed,review cluster summaries and maps,7,strong candidate
3,8,0.024847,0.207717,0.018133,0.207200,8.359796,0.719180,1.947746,0,11.750000,0,6.875000,0.670982,0.638300,0.732880,3.0,0.799475,weak,good,good,good,not assessed,review cluster summaries and maps,6,challenger / needs review
4,9,0.024773,0.200297,0.018075,0.200842,8.085182,0.714873,1.953951,0,11.444444,0,7.000000,0.695718,0.757392,0.806455,1.0,0.913859,weak,good,good,good,not assessed,review cluster summaries and maps,6,challenger / needs review
5,10,0.024668,0.199088,0.017969,0.199534,8.070738,0.737785,1.959723,0,11.900000,0,7.400000,0.739004,0.919085,0.901844,1.0,0.973412,weak,good,good,good,not assessed,review cluster summaries and maps,6,challenger / needs review


## 15. Human review checklist

Use the generated scorecard to narrow the field, then manually inspect the strongest models.

Recommended review sequence:

1. Reject K values with tiny clusters unless the tiny cluster is substantively meaningful.
2. Reject K values where many clusters are near-duplicates.
3. Compare K-to-K lineage tables.
4. For each extra cluster, ask: is this a genuinely new type or just a fine split?
5. Map the challenger models.
6. Pick a main K and one challenger K.
7. Only later test political usefulness once election/member data exists.

Typical interpretation pattern:

| K | likely role |
|---:|---|
| 5 | baseline |
| 6 | likely main challenger |
| 7 | possible if it reveals a real new type |
| 8–10 | likely too granular unless maps and lineage prove otherwise |